# channels

> Build a multi-channel model input from one or more single-channel source images. Generic over the number of source images and the number of output channels.

Many inspection setups capture the same scene under several illuminations (e.g. diffuse + specular). A good segmentation input often stacks the raw captures together with a *derived* channel that makes the target easier to see. This module provides small, composable channel operations and a registry-driven builder.

In [ ]:
#| default_exp channels

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from typing import Callable, Dict, List, Sequence

import cv2
import numpy as np

## A synthetic source pair

We work experiment-first: make two tiny 8-bit images so every cell below runs with no external data. Swap these for `cv2.imread(path, cv2.IMREAD_GRAYSCALE)` on real data.

In [ ]:
rng = np.random.default_rng(0)
ch0 = rng.integers(0, 256, (64, 64), dtype=np.uint8)
ch1 = rng.integers(0, 256, (64, 64), dtype=np.uint8)
print('ch0', ch0.shape, ch0.dtype, '| ch1', ch1.shape, ch1.dtype)

## Channel primitives

Each primitive takes single-channel `uint8` arrays and returns a single-channel `uint8` array. They are the smallest reusable unit: one function, one transformation.

In [ ]:
# experiment: try the candidate derived channels and confirm each stays uint8 and same shape
ld = ((ch1.astype(np.float32) - ch0.astype(np.float32) + 255.) / 2.).clip(0, 255).astype(np.uint8)
pr = ((ch0.astype(np.uint16) * ch1.astype(np.uint16)) // 255).astype(np.uint8)
ad = cv2.absdiff(ch0, ch1)
for name, out in [('linear_diff', ld), ('product', pr), ('abs_diff', ad)]:
    print(f'{name:12s} shape={out.shape} dtype={out.dtype} range=({out.min()},{out.max()})')

In [ ]:
#| export
def linear_diff(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    """Signed difference `b - a` rescaled into `[0, 255]` (128 == no change).

    Example:
        >>> linear_diff(np.zeros((2, 2), np.uint8), np.full((2, 2), 255, np.uint8)).max()
        255
    """
    return ((b.astype(np.float32) - a.astype(np.float32) + 255.) / 2.).clip(0, 255).astype(np.uint8)

def product(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    "Normalised product `a*b/255`: bright only where BOTH inputs are bright."
    return ((a.astype(np.uint16) * b.astype(np.uint16)) // 255).astype(np.uint8)

def abs_diff(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    "Absolute difference `|a - b|`: highlights where the two captures disagree."
    return cv2.absdiff(a, b)

def tophat(x: np.ndarray, k: int = 21) -> np.ndarray:
    "White top-hat: small bright structures (smaller than a `k`x`k` kernel)."
    kern = cv2.getStructuringElement(cv2.MORPH_RECT, (k, k))
    return cv2.morphologyEx(x, cv2.MORPH_TOPHAT, kern)

In [ ]:
# test: primitives preserve shape and dtype
for fn in (linear_diff, product, abs_diff):
    out = fn(ch0, ch1)
    assert out.shape == ch0.shape and out.dtype == np.uint8
assert tophat(ch0, 7).shape == ch0.shape

## Channel-op registry

To build inputs from a config (a list of channel names) we wrap the primitives as ops that consume the *list of source images* and return one channel. This keeps the builder generic: any number of source images, any number of output channels.

In [ ]:
#| export
# An op maps the list of source images -> one single-channel uint8 array.
ChannelOp = Callable[[Sequence[np.ndarray]], np.ndarray]

CHANNEL_OPS: Dict[str, ChannelOp] = {
    'ch0':         lambda imgs: imgs[0],
    'ch1':         lambda imgs: imgs[1],
    'linear_diff': lambda imgs: linear_diff(imgs[0], imgs[1]),
    'product':     lambda imgs: product(imgs[0], imgs[1]),
    'abs_diff':    lambda imgs: abs_diff(imgs[0], imgs[1]),
    'tophat':      lambda imgs: tophat(imgs[1], 21),
}

# Default 3-channel input: the two raw captures plus their product.
DEFAULT_CHANNELS: List[str] = ['ch0', 'ch1', 'product']

In [ ]:
#| export
def stack_channels(
    images: Sequence[np.ndarray],  # source single-channel images, same HxW
    ops: Sequence[ChannelOp],      # ops producing each output channel
) -> np.ndarray:
    """Stack the outputs of `ops` into an `(H, W, C)` uint8 image.

    Lower-level escape hatch when you want custom callables instead of
    registry names.
    """
    channels = [op(images) for op in ops]
    return np.stack(channels, axis=-1).astype(np.uint8)

def build_image(
    images: Sequence[np.ndarray],          # source single-channel images, same HxW
    channel_names: Sequence[str] = DEFAULT_CHANNELS,  # registry keys, one per output channel
) -> np.ndarray:
    """Build an `(H, W, C)` model input from named ops in `CHANNEL_OPS`.

    Example:
        >>> a = np.zeros((4, 4), np.uint8); b = np.full((4, 4), 255, np.uint8)
        >>> build_image([a, b], ['ch0', 'ch1', 'product']).shape
        (4, 4, 3)
    """
    ops = [CHANNEL_OPS[name] for name in channel_names]
    return stack_channels(images, ops)

In [ ]:
# test: default build is 3-channel; a 4-channel multi-spectral build also works
img3 = build_image([ch0, ch1])
assert img3.shape == (64, 64, 3) and img3.dtype == np.uint8
img4 = build_image([ch0, ch1], ['ch0', 'ch1', 'abs_diff', 'product'])
assert img4.shape == (64, 64, 4)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()